In [0]:
   from pyspark.sql import functions as F, Window
   from delta.tables import DeltaTable

   BRONZE = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze"

   dbutils.widgets.text("run_date", "")
   run_date = dbutils.widgets.get("run_date")

   def batch_exists(path):
       try:
           return any(f.name.endswith(".parquet") for f in dbutils.fs.ls(path))
       except Exception as e:
           if "FileNotFound" in str(e) or "PATH_NOT_FOUND" in str(e):
               return False
           raise

   src_path = f"{BRONZE}/orders/ingest_date={run_date}"

   if not batch_exists(src_path):
       print(f"No new orders batch for {run_date}: Silver unchanged.")
   else:
       batch = spark.read.parquet(src_path)
       w = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
       src = (batch.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
                .withColumn("order_date", F.to_date("order_date"))
                .withColumn("order_date",
                    F.when(F.col("order_date") == F.lit("1900-01-01").cast("date"),
                           F.to_date("created_at"))
                     .otherwise(F.col("order_date")))
                .withColumn("order_amount", F.col("order_amount").cast("decimal(12,2)"))
                .filter(F.col("order_id").isNotNull()))

       (DeltaTable.createIfNotExists(spark)
           .tableName("atliq.silver.orders").addColumns(src.schema).execute())

       (DeltaTable.forName(spark, "atliq.silver.orders").alias("t")
           .merge(src.alias("s"), "t.order_id = s.order_id")
           .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
           .whenNotMatchedInsertAll()
           .execute())
       print("orders merged")

orders merged


In [0]:
src_path = f"{BRONZE}/payments/ingest_date={run_date}"

if not batch_exists(src_path):
    print(f"No new payments batch for {run_date}: Silver unchanged.")
else:
    batch = spark.read.parquet(src_path)
    w = Window.partitionBy("payment_id").orderBy(F.col("updated_at").desc())
    src = (batch.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
                .filter(F.col("payment_id").isNotNull()))

    (DeltaTable.createIfNotExists(spark)
        .tableName("atliq.silver.payments").addColumns(src.schema).execute())

    (DeltaTable.forName(spark, "atliq.silver.payments").alias("t")
        .merge(src.alias("s"), "t.payment_id = s.payment_id")
        .whenMatchedUpdateAll(condition="s.updated_at > t.updated_at")
        .whenNotMatchedInsertAll()
        .execute())
    print("payments merged")

payments merged


In [0]:
src_path = f"{BRONZE}/order_items/ingest_date={run_date}"

if not batch_exists(src_path):
    print(f"No new order_items batch for {run_date}: Silver unchanged.")
else:
    batch = spark.read.parquet(src_path)
    w = Window.partitionBy("order_item_id").orderBy(F.col("created_at").desc())
    src = (batch.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
                .filter(F.col("order_item_id").isNotNull()))

    (DeltaTable.createIfNotExists(spark)
        .tableName("atliq.silver.order_items").addColumns(src.schema).execute())

    (DeltaTable.forName(spark, "atliq.silver.order_items").alias("t")
        .merge(src.alias("s"), "t.order_item_id = s.order_item_id")
        .whenNotMatchedInsertAll()
        .execute())
    print("order_items merged")

order_items merged
